In [ ]:
from agentic_rag.rag_helper import RAGBase
from agentic_rag.ingest import load_faq_data, build_index
from gitsource import GithubRepositoryDataReader
from minsearch import Index
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)

True

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

# Q1. How many lesson pages

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
len(documents)

72

# Q2. Indexing and searching

In [ ]:
index = Index(
    text_fields=['content'],
    keyword_fields=['filename'],
)
index.fit(documents)

results = index.search(
    "How does the agentic loop keep calling the model until it stops?",
    num_results=5,
)
print(results[0]['filename'])

01-agentic-rag/lessons/14-agentic-loop.md


# Q3. RAG

In [ ]:
LLM_MODEL = "MiniMax-M2.7"

api_key = os.environ.get("OPENAI_API_KEY")
base_url = os.environ.get("OPENAI_BASE_URL")

if not api_key or not base_url:
    raise SystemExit(
        "Missing env vars. Set OPENAI_API_KEY and OPENAI_BASE_URL in .env."
    )

openai_client = OpenAI(api_key=api_key, base_url=base_url)

In [ ]:
from agentic_rag.rag_helper import LessonRAG

rag = LessonRAG(index=index, llm_client=openai_client, model=LLM_MODEL)
result = rag.rag("How does the agentic loop keep calling the model until it stops?")

tokens_q3 = result.usage.input_tokens
print(tokens_q3)

7125


# Q4. Chunking

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

# Q5. RAG with chunking

In [ ]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
chunk_index.fit(chunks)

chunk_rag = LessonRAG(index=chunk_index, llm_client=openai_client, model=LLM_MODEL)


result = chunk_rag.rag("How does the agentic loop keep calling the model until it stops?")

tokens_q5 = result.usage.input_tokens

print(tokens_q5)

2335


In [17]:
print(tokens_q3 / tokens_q5)

3.051391862955032


# Q6. Turning it into an agent

In [18]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat.interface import StdOutputInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

def search(query: str) -> list[dict]:
    """Search the course lessons for content matching the query."""
    return chunk_index.search(query, num_results=5)

instructions = (
    "You're a course teaching assistant. Answer the student's question "
    "using the search tool. Make multiple searches with different "
    "keywords before answering."
)

In [20]:
agent_tools = Tools()
agent_tools.add_tool(search)

chat_interface = StdOutputInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model=LLM_MODEL, client=openai_client),
)

In [ ]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)


search_calls = sum(
    1 for m in result.all_messages
    if getattr(m, "type", None) == "function_call"
)

print(search_calls)

-> Response received

Assistant: I'll help you understand the agentic loop and how it differs from plain RAG. Let me search for relevant information in the course lessons.



--- Function Call ---
Function: search
Arguments: {"query": "agentic loop"}
Result: [
  {
    "start": 4000,
    "content": "esult.\n\nThe `result` is a `LoopResult` with `all_messages` (the full\nconversation), token counts, and `cost` (computed from token usage).\n\n## Cost and tokens\n\nLook at what the call cost:\n\n```python\nresult.cost\n```\n\nUseful while developing - especially with multi-turn agents where one\nprompt can trigger several model calls. The handwritten loop made you\ncompute this by hand. The framework keeps a running total for you.\n\nYou can also look at the full message history.\n\n```python\nresult.all_messages\n```\n\nThis is just a list - the same `messages` list we maintained by hand.\n\n## Continuing the conversation\n\nTake the messages from the previous result and pass them as\n`pr